# Day05 — error analysis

## D5-01 — проверенный validation reference

Цель: загрузить сохранённые predictions выбранного Day04 candidate без fit, CV или оценки test. Входы — Day02 processed/provenance, Day03 manifest/reference, Day04 selection/config/predictions/metrics/provenance.

SHA-256 проверяет байты artifacts, а semantic checks — IDs, порядок, labels и параметры. Join выполняется one-to-one по `source_row_id`: одинаковая длина таблиц не гарантирует соответствия предложений. Все validation строки сохраняются; `errors` — только фильтр, не новая выборка для настройки модели.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/finnews_sentiment").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/finnews_sentiment").is_dir():
    raise RuntimeError("Run from the project root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.models.error_analysis import load_validation_analysis

analysis = load_validation_analysis(ROOT)
print("Candidate:", analysis.candidate_id)
print("Selection:", analysis.selection_path.relative_to(ROOT))
print("Parameters:", analysis.parameters)

Candidate: linear_svc
Selection: reports/day04/selection.json
Parameters: {'tfidf': {'max_features': 5000, 'ngram_range': [1, 2], 'analyzer': 'word', 'lowercase': False, 'stop_words': None, 'min_df': 1, 'max_df': 1.0, 'use_idf': True, 'smooth_idf': True, 'sublinear_tf': False, 'norm': 'l2', 'token_pattern': '(?u)\\b\\w\\w+\\b', 'strip_accents': None}, 'classifier': {'C': 1.0, 'class_weight': None, 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 5000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}}


In [2]:
n_valid = len(analysis.table)
n_errors = len(analysis.errors)
print("Table shape:", analysis.table.shape)
print(f"N_valid={n_valid}; errors={n_errors}; error rate={n_errors / n_valid:.2%}")
display(analysis.table.head())
display(analysis.errors.head())

Table shape: (553, 9)
N_valid=553; errors=84; error rate=15.19%


,source_row_id,group_id,text,text_clean,true_label_id,pred_label_id,true_label,pred_label,is_error
0,2,2,"For the last quarter of 2010 , Componenta 's n...","for the last quarter of 2010 , componenta 's n...",2,2,positive,positive,False
1,3,3,"In the third quarter of 2010 , net sales incre...","in the third quarter of 2010 , net sales incre...",2,2,positive,positive,False
2,11,11,Consolidated net sales increased 16 % to reach...,consolidated net sales increased 16 % to reach...,2,2,positive,positive,False
3,16,16,"Lifetree was founded in 2000 , and its revenue...","lifetree was founded in 2000 , and its revenue...",2,1,positive,neutral,True
4,28,28,The fair value of the property portfolio doubl...,the fair value of the property portfolio doubl...,2,1,positive,neutral,True


,source_row_id,group_id,text,text_clean,true_label_id,pred_label_id,true_label,pred_label,is_error
3,16,16,"Lifetree was founded in 2000 , and its revenue...","lifetree was founded in 2000 , and its revenue...",2,1,positive,neutral,True
4,28,28,The fair value of the property portfolio doubl...,the fair value of the property portfolio doubl...,2,1,positive,neutral,True
8,59,59,The combination of all services enabling us to...,the combination of all services enabling us to...,2,1,positive,neutral,True
10,72,72,Biohit already services many current Genesis c...,biohit already services many current genesis c...,2,1,positive,neutral,True
20,114,114,Finnish Okmetic that manufactures and processe...,finnish okmetic that manufactures and processe...,2,1,positive,neutral,True


### Интерпретация и самопроверка

Таблица `(N_valid, 9)` сохраняет raw/clean text и исходный manifest order. `pred_label_id` явно получен из `selected_pred_label_id`. Показаны первые реальные ошибки в исходном порядке; это проверка загрузки, ручной разбор остаётся D5-03. Доля ошибок равна `1 − accuracy`, но не заменяет macro F1: частый neutral может доминировать в accuracy. Нулевое число ошибок допустимо.

1. Почему одинаковой длины таблиц недостаточно для alignment?
2. Почему переставленные reference predictions вызывают ошибку, хотя join по ID технически возможен?
3. Какие проверки защищают от изменения данных и от пересечения text groups?
4. Почему найденные ошибки не дают права менять текущую модель перед test?